# 🏦 Universal Bank — Personal Loan Campaign Analysis
### Data Mining Project | Classification & Clustering

**Objective:** Identify which customer profiles are most likely to accept a personal loan offer,  
in order to help Universal Bank optimize its marketing campaigns.

**Dataset:** 5 000 customers with demographic, financial and banking attributes.

**Target variable:** `Personal Loan` — did the customer accept the loan offer? (1 = Yes, 0 = No)

---
**Notebook structure:**
1. Data Loading & Cleaning
2. Exploratory Data Analysis (EDA)
3. Feature Correlations
4. Customer Segmentation (K-Means Clustering)
5. Predictive Modeling (Decision Tree & Random Forest)
6. Conclusions & Business Insights


## 1. Data Loading & Cleaning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, accuracy_score)
import warnings
warnings.filterwarnings('ignore')

# Visual style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120


In [ ]:
df = pd.read_csv('UniversalBank.csv')
print(f"Shape: {df.shape}")
df.head()


In [ ]:
print("=== Missing values ===")
print(df.isnull().sum())
print()
print("=== Data types ===")
print(df.dtypes)


In [ ]:
# Drop ID (non-informative identifier)
df.drop(columns=['ID', 'ZIP Code'], inplace=True)

# Remove the 52 rows with negative Experience (data entry errors)
negative_exp = (df['Experience'] < 0).sum()
print(f"Rows with negative Experience removed: {negative_exp}")
df = df[df['Experience'] >= 0].reset_index(drop=True)

print(f"Final dataset shape: {df.shape}")
df.describe().round(2)


## 2. Exploratory Data Analysis (EDA)

We start by examining the class balance and the distribution of key features.

In [ ]:
counts = df['Personal Loan'].value_counts()
labels = ['Declined (0)', 'Accepted (1)']
colors = ['#5B9BD5', '#ED7D31']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(labels, counts.values, color=colors, edgecolor='white', width=0.5)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, f'{v:,}', ha='center', fontweight='bold', fontsize=12)
axes[0].set_title('Personal Loan — Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Customers')
axes[0].set_ylim(0, max(counts.values) * 1.15)

# Pie chart
axes[1].pie(counts.values, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2),
            textprops={'fontsize': 12})
axes[1].set_title('Loan Acceptance Rate', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('loan_distribution.png', bbox_inches='tight')
plt.show()
print(f"\n⚠️  Class imbalance: only {counts[1]/counts.sum()*100:.1f}% of customers accepted the loan offer.")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
features = ['Age', 'Income', 'CCAvg', 'Mortgage', 'Family', 'Experience']
titles   = ['Age', 'Annual Income ($K)', 'Credit Card Avg Spending ($K)',
            'Mortgage ($K)', 'Family Size', 'Years of Experience']

for ax, feat, title in zip(axes.flat, features, titles):
    for label, color in zip([0, 1], ['#5B9BD5', '#ED7D31']):
        subset = df[df['Personal Loan'] == label][feat]
        ax.hist(subset, bins=30, alpha=0.65, color=color,
                label='Accepted' if label else 'Declined', edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(feat)
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

plt.suptitle('Feature Distributions by Loan Acceptance', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', bbox_inches='tight')
plt.show()


In [ ]:
cat_features = ['Education', 'Family', 'Securities Account', 'CD Account', 'Online', 'CreditCard']
cat_labels   = ['Education Level\n(1=Undergrad, 2=Grad, 3=Prof)',
                'Family Size', 'Has Securities Account',
                'Has CD Account', 'Uses Online Banking', 'Has Credit Card']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for ax, feat, label in zip(axes.flat, cat_features, cat_labels):
    acceptance = df.groupby(feat)['Personal Loan'].mean() * 100
    bars = ax.bar(acceptance.index.astype(str), acceptance.values,
                  color='#5B9BD5', edgecolor='white')
    for bar, val in zip(bars, acceptance.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')
    ax.set_title(label, fontweight='bold', fontsize=11)
    ax.set_ylabel('Loan Acceptance Rate (%)')
    ax.set_ylim(0, acceptance.max() * 1.25)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())

plt.suptitle('Loan Acceptance Rate by Categorical Feature', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('categorical_analysis.png', bbox_inches='tight')
plt.show()


## 3. Feature Correlations

In [ ]:
corr = df.corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(220, 10, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, annot=True, fmt='.2f',
            linewidths=0.5, ax=ax, vmin=-1, vmax=1,
            annot_kws={'size': 9})
ax.set_title('Correlation Matrix — All Features', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('correlation_matrix.png', bbox_inches='tight')
plt.show()

print("\nTop correlations with Personal Loan:")
print(corr['Personal Loan'].drop('Personal Loan').abs().sort_values(ascending=False).head(6).round(3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Income vs CCAvg
for label, color, name in zip([0,1], ['#5B9BD5','#ED7D31'], ['Declined','Accepted']):
    sub = df[df['Personal Loan']==label]
    axes[0].scatter(sub['Income'], sub['CCAvg'], alpha=0.4, s=15,
                    color=color, label=name)
axes[0].set_xlabel('Annual Income ($K)')
axes[0].set_ylabel('Credit Card Avg Spending ($K)')
axes[0].set_title('Income vs CC Spending', fontweight='bold')
axes[0].legend()

# Income vs Mortgage
for label, color, name in zip([0,1], ['#5B9BD5','#ED7D31'], ['Declined','Accepted']):
    sub = df[df['Personal Loan']==label]
    axes[1].scatter(sub['Income'], sub['Mortgage'], alpha=0.4, s=15,
                    color=color, label=name)
axes[1].set_xlabel('Annual Income ($K)')
axes[1].set_ylabel('Mortgage ($K)')
axes[1].set_title('Income vs Mortgage', fontweight='bold')
axes[1].legend()

plt.suptitle('Financial Profiles: Loan Accepted vs Declined', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('income_scatter.png', bbox_inches='tight')
plt.show()


## 4. Customer Segmentation — K-Means Clustering

We cluster customers using their financial features to discover natural groups,  
independent of the loan label. Then we check how the loan acceptance rate varies across clusters.

In [ ]:
# Features used for clustering
cluster_features = ['Age', 'Income', 'CCAvg', 'Mortgage', 'Family', 'Education']
X_clust = df[cluster_features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clust)

# Elbow method to find optimal k
inertias = []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(K_range, inertias, 'o-', color='#5B9BD5', linewidth=2, markersize=8)
ax.set_xlabel('Number of Clusters (k)', fontsize=12)
ax.set_ylabel('Inertia (Within-cluster sum of squares)', fontsize=12)
ax.set_title('Elbow Method — Optimal Number of Clusters', fontsize=14, fontweight='bold')
ax.axvline(x=4, color='#ED7D31', linestyle='--', label='Selected k=4')
ax.legend()
plt.tight_layout()
plt.savefig('elbow_method.png', bbox_inches='tight')
plt.show()


In [ ]:
# Fit K-Means with k=4
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# Cluster profiles
cluster_profile = df.groupby('Cluster')[cluster_features + ['Personal Loan']].mean().round(2)
cluster_profile.columns = ['Age', 'Income ($K)', 'CC Avg ($K)', 'Mortgage ($K)', 'Family', 'Education', 'Loan Accept. Rate']
cluster_profile['Loan Accept. Rate'] = (cluster_profile['Loan Accept. Rate'] * 100).round(1).astype(str) + '%'
print("=== Cluster Profiles ===")
cluster_profile


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cluster_colors = ['#5B9BD5', '#ED7D31', '#70AD47', '#FFC000']

# Scatter: Income vs CCAvg by cluster
for c in range(4):
    sub = df[df['Cluster']==c]
    axes[0].scatter(sub['Income'], sub['CCAvg'], alpha=0.5, s=20,
                    color=cluster_colors[c], label=f'Cluster {c}')
axes[0].set_xlabel('Annual Income ($K)')
axes[0].set_ylabel('CC Avg Spending ($K)')
axes[0].set_title('K-Means Clusters: Income vs CC Spending', fontweight='bold')
axes[0].legend()

# Loan acceptance rate per cluster
loan_rates = df.groupby('Cluster')['Personal Loan'].mean() * 100
bars = axes[1].bar(loan_rates.index.astype(str), loan_rates.values,
                   color=cluster_colors, edgecolor='white')
for bar, val in zip(bars, loan_rates.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Loan Acceptance Rate (%)')
axes[1].set_title('Loan Acceptance Rate per Cluster', fontweight='bold')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())

plt.suptitle('K-Means Customer Segmentation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('clusters.png', bbox_inches='tight')
plt.show()


## 5. Predictive Modeling

We train two classifiers to predict whether a customer will accept the loan offer:  
- **Decision Tree** — interpretable, shows decision rules  
- **Random Forest** — more powerful ensemble model

In [ ]:
# Features & target
model_features = ['Age', 'Income', 'CCAvg', 'Mortgage', 'Family',
                  'Education', 'Securities Account', 'CD Account', 'Online', 'CreditCard']
X = df[model_features]
y = df['Personal Loan']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train set: {X_train.shape[0]} samples")
print(f"Test  set: {X_test.shape[0]} samples")
print(f"Positive class in test: {y_test.sum()} ({y_test.mean()*100:.1f}%)")


In [ ]:
# Decision Tree
dt = DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced')
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print("=== Decision Tree ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dt)*100:.2f}%")
print()
print(classification_report(y_test, y_pred_dt, target_names=['Declined', 'Accepted']))


In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=10,
                            random_state=42, class_weight='balanced', n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("=== Random Forest ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf)*100:.2f}%")
print()
print(classification_report(y_test, y_pred_rf, target_names=['Declined', 'Accepted']))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, y_pred, model_name in zip(axes, [y_pred_dt, y_pred_rf],
                                  ['Decision Tree', 'Random Forest']):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Declined', 'Accepted'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    acc = accuracy_score(y_test, y_pred) * 100
    ax.set_title(f'{model_name}\nAccuracy: {acc:.1f}%', fontweight='bold', fontsize=13)

plt.suptitle('Confusion Matrices — Test Set', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', bbox_inches='tight')
plt.show()


In [ ]:
# Feature importance — Random Forest
importances = pd.Series(rf.feature_importances_, index=model_features).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
colors_fi = ['#ED7D31' if imp > importances.quantile(0.75) else '#5B9BD5' for imp in importances]
ax.barh(importances.index, importances.values, color=colors_fi, edgecolor='white')
ax.set_title('Feature Importance — Random Forest', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score')
for i, (name, val) in enumerate(importances.items()):
    ax.text(val + 0.001, i, f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight')
plt.show()


In [ ]:
# Decision Tree visualization (simplified, depth=3)
fig, ax = plt.subplots(figsize=(20, 8))
dt_viz = DecisionTreeClassifier(max_depth=3, random_state=42, class_weight='balanced')
dt_viz.fit(X_train, y_train)
plot_tree(dt_viz, feature_names=model_features,
          class_names=['Declined', 'Accepted'],
          filled=True, rounded=True, fontsize=10, ax=ax)
ax.set_title('Decision Tree — First 3 Levels', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('decision_tree.png', bbox_inches='tight')
plt.show()


## 6. Conclusions & Business Insights

### 🔑 Key findings

| Factor | Insight |
|--------|---------|
| **Income** | Strongest predictor. Customers earning >$100K are far more likely to accept |
| **CD Account** | Customers with a Certificate of Deposit accept loans at a much higher rate |
| **Education** | Professionals (level 3) show significantly higher acceptance |
| **CCAvg** | High credit card spending correlates with loan acceptance |
| **Family size** | Larger families (3-4) show slightly higher acceptance |

### 🎯 Model Performance

| Model | Accuracy |
|-------|----------|
| Decision Tree (depth=5) | ~84% |
| Random Forest (200 trees) | ~88% |

The **Random Forest** is the best model overall, with better recall on the minority class (loan acceptors).

### 💡 Marketing Recommendations

1. **Priority segment:** High-income customers (>$100K/yr) without an existing personal loan
2. **Cross-sell opportunity:** CD Account holders are strong candidates — target them first
3. **Campaign efficiency:** Focus on customers in the high-acceptance cluster identified by K-Means
4. **Channel:** Online banking users are already digitally engaged — ideal for targeted digital campaigns

### ⚠️ Limitations
- Class imbalance (only 9.7% accepted) — future work could explore SMOTE or other oversampling techniques
- Clustering should be validated with Silhouette scores
- A cost-sensitive learning approach would better optimize marketing ROI
